# 🔧 Fine-tuning Qwen3-VL-8B: Bản Vẽ Kỹ Thuật → Mô Tả Text

**Mục tiêu:** Fine-tune model Vision-Language để nhận đầu vào là ảnh bản vẽ kỹ thuật và xuất ra mô tả text chi tiết về bản vẽ đó.

**Cấu trúc thư mục trên Google Drive:**
```
MyDrive/TechDrawing_FineTune/
├── dataset/
│   ├── images/           ← Bỏ ảnh bản vẽ kỹ thuật vào đây (.jpg, .png)
│   └── annotations.json  ← File nhãn (tạo tự động bởi cell bên dưới)
├── outputs/              ← Checkpoint training (tự động)
└── saved_model/          ← Model đã fine-tune (tự động)
```

**Format annotations.json:**
```json
[
  {
    "image": "drawing_001.jpg",
    "description": "Bản vẽ kỹ thuật thể hiện chi tiết trục bậc..."
  }
]
```

---
**Powered by [Unsloth](https://unsloth.ai/) | Model: unsloth/Qwen3-VL-8B-Instruct-unsloth-bnb-4bit**

## Bước 1: Cài đặt thư viện

In [ ]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
!pip install transformers==4.57.1
!pip install --no-deps trl==0.22.2
!pip install Pillow

## Bước 2: Mount Google Drive & Thiết lập đường dẫn

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# ============================================================
# ⚙️  CẤU HÌNH - Chỉnh sửa nếu cần
# ============================================================
BASE_DIR   = "/content/drive/MyDrive/TechDrawing_FineTune"
IMAGES_DIR = f"{BASE_DIR}/dataset/images"
ANNOT_FILE = f"{BASE_DIR}/dataset/annotations.json"
OUTPUT_DIR = f"{BASE_DIR}/outputs"
SAVE_DIR   = f"{BASE_DIR}/saved_model/qwen_techdrawing_lora"

HF_USERNAME  = "YOUR_HF_USERNAME"    # ← Điền Hugging Face username của bạn
HF_TOKEN     = "YOUR_HF_TOKEN"       # ← Điền HF token (https://huggingface.co/settings/tokens)
HF_REPO_NAME = "qwen3-vl-8b-techdrawing-lora"  # ← Tên repo trên HF
# ============================================================

# Tạo thư mục nếu chưa có
for d in [IMAGES_DIR, OUTPUT_DIR, SAVE_DIR]:
    os.makedirs(d, exist_ok=True)

print("✅ Google Drive đã mount thành công!")
print(f"📁 Thư mục gốc    : {BASE_DIR}")
print(f"🖼️  Thư mục ảnh    : {IMAGES_DIR}")
print(f"📝 File annotations: {ANNOT_FILE}")
print(f"💾 Lưu model tại  : {SAVE_DIR}")

## Bước 3: Tạo Dữ Liệu Mẫu (Chạy 1 lần để Bootstrap)

> **Nếu bạn đã có `annotations.json`, hãy bỏ qua cell này.**  
> Cell này tạo file mẫu để bạn biết format cần điền. Sau đó bạn:
> 1. Bỏ ảnh bản vẽ thực vào `IMAGES_DIR`
> 2. Cập nhật `annotations.json` với mô tả thực tế

In [ ]:
import json
from PIL import Image, ImageDraw, ImageFont
import numpy as np

# ----- Tạo ảnh bản vẽ kỹ thuật GIẢ để minh họa -----
def create_dummy_technical_drawing(filename, title, shape='shaft'):
    """Tạo ảnh bản vẽ kỹ thuật đơn giản để minh họa format."""
    img = Image.new('RGB', (640, 480), color=(255, 255, 255))
    draw = ImageDraw.Draw(img)

    # Khung viền tiêu chuẩn
    draw.rectangle([10, 10, 630, 470], outline='black', width=2)
    draw.rectangle([10, 10, 630, 50], outline='black', width=2)

    # Tiêu đề
    draw.text((20, 20), title, fill='black')

    if shape == 'shaft':
        # Vẽ trục bậc đơn giản
        draw.rectangle([100, 180, 540, 300], outline='black', width=2)
        draw.rectangle([250, 150, 390, 330], outline='black', width=2)
        # Đường kích thước
        draw.line([100, 320, 540, 320], fill='black', width=1)
        draw.text((300, 325), "L = 440mm", fill='black')
        draw.line([570, 180, 570, 300], fill='black', width=1)
        draw.text((575, 230), "Ø30", fill='black')
        draw.text((215, 335), "Ø45", fill='black')

    elif shape == 'plate':
        # Vẽ tấm đục lỗ
        draw.rectangle([80, 120, 560, 390], outline='black', width=2)
        for x in [160, 300, 440]:
            for y in [200, 310]:
                draw.ellipse([x-20, y-20, x+20, y+20], outline='black', width=2)
        draw.text((200, 410), "6 lỗ Ø15, bố trí đều", fill='black')

    elif shape == 'bracket':
        # Vẽ bracket
        draw.polygon([(150,350),(150,150),(200,150),(200,250),(500,250),(500,350)], outline='black')
        draw.text((280, 270), "Góc vuông 90°", fill='black')

    img.save(filename)
    print(f"  ✅ Đã tạo: {filename}")

# Kiểm tra đã có annotations.json chưa
if os.path.exists(ANNOT_FILE):
    print(f"⚠️  File annotations.json đã tồn tại tại {ANNOT_FILE}")
    print("   Bỏ qua việc tạo dữ liệu mẫu. Xóa file nếu muốn tạo lại.")
else:
    print("📦 Tạo dữ liệu mẫu minh họa...")

    samples = [
        {
            "image": "drawing_001.jpg",
            "description": (
                "Bản vẽ kỹ thuật thể hiện chi tiết trục bậc có hai bậc đường kính. "
                "Đường kính thân trục chính là Ø30mm, chiều dài tổng thể L=440mm. "
                "Phần bậc giữa có đường kính Ø45mm, chiều dài 140mm, bố trí đối xứng. "
                "Vật liệu: Thép C45. Độ nhám bề mặt Ra=1.6. Dung sai lắp ghép H7/k6 tại hai đầu trục."
            ),
            "shape": "shaft"
        },
        {
            "image": "drawing_002.jpg",
            "description": (
                "Bản vẽ kỹ thuật thể hiện tấm đệm hình chữ nhật có kích thước 480x270mm. "
                "Tấm được gia công 6 lỗ thông Ø15mm, bố trí đều thành 2 hàng 3 cột, "
                "khoảng cách trục lỗ 140mm theo chiều dài và 110mm theo chiều rộng. "
                "Chiều dày tấm 10mm. Vật liệu: Thép CT3. Tất cả mép sắc cần vát mép 1×45°."
            ),
            "shape": "plate"
        },
        {
            "image": "drawing_003.jpg",
            "description": (
                "Bản vẽ kỹ thuật thể hiện giá đỡ hình chữ L (bracket). "
                "Cánh đứng cao 200mm, dày 50mm. Cánh nằm ngang dài 350mm, dày 100mm. "
                "Góc giữa hai cánh là 90°. Vật liệu: Gang xám GX150. "
                "Bề mặt lắp ghép cần mài phẳng đạt Ra=3.2."
            ),
            "shape": "bracket"
        },
    ]

    # Tạo ảnh mẫu
    for s in samples:
        img_path = os.path.join(IMAGES_DIR, s["image"])
        if not os.path.exists(img_path):
            create_dummy_technical_drawing(img_path, s["image"], s["shape"])

    # Tạo annotations.json (không có field 'shape')
    annotations = [{"image": s["image"], "description": s["description"]} for s in samples]
    with open(ANNOT_FILE, 'w', encoding='utf-8') as f:
        json.dump(annotations, f, ensure_ascii=False, indent=2)

    print(f"\n✅ Đã tạo {len(samples)} mẫu dữ liệu!")
    print(f"📝 File annotations: {ANNOT_FILE}")
    print(f"🖼️  Thư mục ảnh    : {IMAGES_DIR}")
    print("\n" + "="*60)
    print("📌 HƯỚNG DẪN TIẾP THEO:")
    print("  1. Bỏ ảnh bản vẽ kỹ thuật thực vào:", IMAGES_DIR)
    print("  2. Mở và chỉnh sửa annotations.json trên Drive")
    print("  3. Chạy tiếp các cell bên dưới để train")
    print("="*60)

## Bước 4: Xem Trước Dữ Liệu

In [ ]:
import json
from PIL import Image
import matplotlib.pyplot as plt
import os

# Load annotations
with open(ANNOT_FILE, 'r', encoding='utf-8') as f:
    annotations = json.load(f)

print(f"📊 Tổng số mẫu: {len(annotations)}")
print("\n--- Danh sách mẫu ---")
for i, ann in enumerate(annotations):
    img_path = os.path.join(IMAGES_DIR, ann['image'])
    exists = '✅' if os.path.exists(img_path) else '❌ THIẾU'
    print(f"[{i+1}] {ann['image']} {exists}")
    print(f"    Mô tả: {ann['description'][:80]}...")
    print()

# Hiển thị ảnh đầu tiên
first_img_path = os.path.join(IMAGES_DIR, annotations[0]['image'])
if os.path.exists(first_img_path):
    fig, axes = plt.subplots(1, min(3, len(annotations)), figsize=(15, 5))
    if len(annotations) == 1:
        axes = [axes]
    for ax, ann in zip(axes, annotations[:3]):
        img_path = os.path.join(IMAGES_DIR, ann['image'])
        if os.path.exists(img_path):
            img = Image.open(img_path)
            ax.imshow(img)
            ax.set_title(ann['image'], fontsize=10)
            ax.set_xlabel(ann['description'][:50] + '...', fontsize=7, wrap=True)
            ax.axis('off')
    plt.tight_layout()
    plt.show()
else:
    print("⚠️  Không tìm thấy ảnh. Hãy đảm bảo đã bỏ ảnh vào IMAGES_DIR.")

## Bước 5: Load Model Qwen3-VL-8B

In [ ]:
from unsloth import FastVisionModel
import torch

model, tokenizer = FastVisionModel.from_pretrained(
    "unsloth/Qwen3-VL-8B-Instruct-unsloth-bnb-4bit",
    load_in_4bit = True,                    # 4bit giảm VRAM, dùng False cho 16bit LoRA
    use_gradient_checkpointing = "unsloth", # Tối ưu bộ nhớ cho context dài
)

print("✅ Đã load model thành công!")

## Bước 6: Cấu hình LoRA Adapters

In [ ]:
model = FastVisionModel.get_peft_model(
    model,
    finetune_vision_layers     = True,  # Fine-tune vision encoder
    finetune_language_layers   = True,  # Fine-tune language decoder
    finetune_attention_modules = True,
    finetune_mlp_modules       = True,

    r            = 16,     # Rank: cao hơn = chính xác hơn nhưng tốn VRAM hơn
    lora_alpha   = 16,     # Khuyến nghị = r
    lora_dropout = 0,
    bias         = "none",
    random_state = 3407,
    use_rslora   = False,
    loftq_config = None,
)

print("✅ LoRA adapters đã được cấu hình!")

## Bước 7: Chuẩn Bị Dataset

In [ ]:
import json
from PIL import Image
import os

# Instruction hướng dẫn model
INSTRUCTION = (
    "Describe this technical drawing briefly and naturally, "
    "as if writing a short description for a 3D CAD file."
)

def load_and_convert_dataset(annotations, images_dir, instruction):
    """Load ảnh và chuyển sang format hội thoại cho VLM."""
    dataset = []
    skipped = 0

    for ann in annotations:
        img_path = os.path.join(images_dir, ann['image'])
        if not os.path.exists(img_path):
            print(f"⚠️  Bỏ qua (không tìm thấy ảnh): {img_path}")
            skipped += 1
            continue

        try:
            image = Image.open(img_path).convert('RGB')
        except Exception as e:
            print(f"⚠️  Lỗi đọc ảnh {ann['image']}: {e}")
            skipped += 1
            continue

        conversation = [
            {
                "role": "user",
                "content": [
                    {"type": "image", "image": image},
                    {"type": "text",  "text":  instruction},
                ]
            },
            {
                "role": "assistant",
                "content": [
                    {"type": "text", "text": ann['description']}
                ]
            },
        ]
        dataset.append({"messages": conversation})

    print(f"✅ Dataset sẵn sàng: {len(dataset)} mẫu hợp lệ, {skipped} mẫu bỏ qua.")
    return dataset

# Load annotations
with open(ANNOT_FILE, 'r', encoding='utf-8') as f:
    annotations = json.load(f)

converted_dataset = load_and_convert_dataset(annotations, IMAGES_DIR, INSTRUCTION)

if len(converted_dataset) == 0:
    raise ValueError("❌ Không có mẫu nào hợp lệ! Hãy kiểm tra lại IMAGES_DIR và annotations.json.")

# Preview
print("\n--- Preview mẫu đầu tiên ---")
sample = converted_dataset[0]
print(f"User text   : {sample['messages'][0]['content'][1]['text'][:60]}...")
print(f"Assistant   : {sample['messages'][1]['content'][0]['text'][:80]}...")

## Bước 8: Thử Inference TRƯỚC Khi Train
*Để thấy sự khác biệt trước/sau fine-tuning*

In [ ]:
from transformers import TextStreamer

FastVisionModel.for_inference(model)

# Lấy ảnh đầu tiên để test
test_image = converted_dataset[0]['messages'][0]['content'][0]['image']

messages = [
    {"role": "user", "content": [
        {"type": "image"},
        {"type": "text", "text": INSTRUCTION}
    ]}
]

input_text = tokenizer.apply_chat_template(messages, add_generation_prompt=True)
inputs = tokenizer(
    test_image,
    input_text,
    add_special_tokens=False,
    return_tensors="pt",
).to("cuda")

print("🤖 Output TRƯỚC fine-tuning:")
print("-" * 50)
text_streamer = TextStreamer(tokenizer, skip_prompt=True)
_ = model.generate(
    **inputs,
    streamer=text_streamer,
    max_new_tokens=256,
    use_cache=True,
    temperature=1.5,
    min_p=0.1
)

## Bước 9: Huấn Luyện Model

> **Gợi ý `max_steps`:**
> - Dataset nhỏ (< 50 ảnh): dùng `num_train_epochs=3` thay cho `max_steps`
> - Dataset vừa (50-500): `max_steps=100`
> - Dataset lớn (> 500): `num_train_epochs=1`

In [ ]:
from unsloth.trainer import UnslothVisionDataCollator
from trl import SFTTrainer, SFTConfig

FastVisionModel.for_training(model)  # Bật chế độ training

# ============================================================
# ⚙️  HYPERPARAMETERS - Điều chỉnh theo nhu cầu
# ============================================================
NUM_SAMPLES = len(converted_dataset)

trainer = SFTTrainer(
    model        = model,
    tokenizer    = tokenizer,
    data_collator = UnslothVisionDataCollator(model, tokenizer),
    train_dataset = converted_dataset,
    args = SFTConfig(
        per_device_train_batch_size    = 1,    # Giảm xuống 1 nếu OOM
        gradient_accumulation_steps    = 4,    # Effective batch = 1×4 = 4
        warmup_steps                   = max(1, NUM_SAMPLES // 10),
        num_train_epochs               = 3,    # Số epoch (tắt max_steps nếu dùng cái này)
        # max_steps                    = 60,  # Hoặc dùng max_steps cố định
        learning_rate                  = 2e-4,
        logging_steps                  = 1,
        optim                          = "adamw_8bit",
        weight_decay                   = 0.01,
        lr_scheduler_type              = "cosine",
        seed                           = 3407,
        output_dir                     = OUTPUT_DIR,
        report_to                      = "none",
        save_strategy                  = "epoch",   # Lưu checkpoint mỗi epoch
        save_total_limit               = 2,          # Giữ tối đa 2 checkpoint

        # Bắt buộc cho vision finetuning:
        remove_unused_columns          = False,
        dataset_text_field             = "",
        dataset_kwargs                 = {"skip_prepare_dataset": True},
        max_length                     = 2048,
    ),
)
# ============================================================

print(f"🚀 Bắt đầu training với {NUM_SAMPLES} mẫu...")
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory       = round(gpu_stats.total_memory  / 1024 / 1024 / 1024, 3)
print(f"GPU: {gpu_stats.name} | VRAM tổng: {max_memory} GB | VRAM đang dùng: {start_gpu_memory} GB")

trainer_stats = trainer.train()

In [ ]:
# Thống kê sau training
used_memory              = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora     = round(used_memory - start_gpu_memory, 3)
used_percentage          = round(used_memory / max_memory * 100, 3)
lora_percentage          = round(used_memory_for_lora / max_memory * 100, 3)

print(f"⏱️  Thời gian training : {round(trainer_stats.metrics['train_runtime']/60, 2)} phút")
print(f"💾 VRAM đỉnh           : {used_memory} GB ({used_percentage}% tổng)")
print(f"📈 VRAM dùng cho LoRA  : {used_memory_for_lora} GB ({lora_percentage}%)")

## Bước 10: Thử Inference SAU Khi Train
*So sánh với kết quả ở Bước 8*

In [ ]:
from transformers import TextStreamer

FastVisionModel.for_inference(model)

test_image = converted_dataset[0]['messages'][0]['content'][0]['image']

messages = [
    {"role": "user", "content": [
        {"type": "image"},
        {"type": "text", "text": INSTRUCTION}
    ]}
]
input_text = tokenizer.apply_chat_template(messages, add_generation_prompt=True)
inputs = tokenizer(
    test_image,
    input_text,
    add_special_tokens=False,
    return_tensors="pt",
).to("cuda")

print("🤖 Output SAU fine-tuning:")
print("-" * 50)
text_streamer = TextStreamer(tokenizer, skip_prompt=True)
_ = model.generate(
    **inputs,
    streamer=text_streamer,
    max_new_tokens=512,
    use_cache=True,
    temperature=0.7,   # Thấp hơn = ổn định hơn khi inference thực tế
    min_p=0.1
)

# Test ảnh mới (ảnh bên ngoài tập train)
print("\n" + "="*60)
print("📷 Test với ảnh tùy chỉnh:")
CUSTOM_IMAGE_PATH = "/content/drive/MyDrive/TechDrawing_FineTune/dataset/images/drawing_001.jpg"
# ↑ Thay bằng đường dẫn ảnh bản vẽ bạn muốn test

if os.path.exists(CUSTOM_IMAGE_PATH):
    custom_image = Image.open(CUSTOM_IMAGE_PATH).convert('RGB')
    messages_custom = [
        {"role": "user", "content": [
            {"type": "image"},
            {"type": "text", "text": INSTRUCTION}
        ]}
    ]
    input_text2 = tokenizer.apply_chat_template(messages_custom, add_generation_prompt=True)
    inputs2 = tokenizer(
        custom_image, input_text2,
        add_special_tokens=False, return_tensors="pt",
    ).to("cuda")
    _ = model.generate(**inputs2, streamer=text_streamer, max_new_tokens=512,
                       use_cache=True, temperature=0.7, min_p=0.1)
else:
    print(f"⚠️  Không tìm thấy '{CUSTOM_IMAGE_PATH}'. Hãy thay bằng đường dẫn ảnh thật.")

## Bước 11: Lưu Model
### 11a. Lưu LoRA Adapters về Google Drive

In [ ]:
# Lưu LoRA adapters vào Google Drive
print(f"💾 Đang lưu LoRA adapters vào Drive: {SAVE_DIR}")
model.save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)
print(f"✅ Đã lưu xong! Đường dẫn: {SAVE_DIR}")

# Liệt kê các file đã lưu
saved_files = os.listdir(SAVE_DIR)
print(f"\n📁 Các file đã lưu ({len(saved_files)} files):")
for f in saved_files:
    size = os.path.getsize(os.path.join(SAVE_DIR, f)) / (1024*1024)
    print(f"  {f} ({size:.1f} MB)")

### 11b. Đẩy LoRA Adapters lên Hugging Face Hub

In [ ]:
# Push LoRA adapters lên Hugging Face Hub
HF_REPO_ID = f"{HF_USERNAME}/{HF_REPO_NAME}"

print(f"🚀 Đang push lên Hugging Face: {HF_REPO_ID}")
model.push_to_hub(
    HF_REPO_ID,
    token=HF_TOKEN,
    private=True,   # Đặt False nếu muốn public
)
tokenizer.push_to_hub(
    HF_REPO_ID,
    token=HF_TOKEN,
)
print(f"✅ Đã push xong! Xem tại: https://huggingface.co/{HF_REPO_ID}")

### 11c. (Tùy chọn) Lưu Model Full 16-bit về Drive
*Dùng để deploy với vLLM hoặc llama.cpp. Kích thước lớn hơn (~16GB).*

In [ ]:
SAVE_16BIT = False  # ← Đặt True để lưu merged 16-bit

if SAVE_16BIT:
    MERGED_DIR = f"{BASE_DIR}/saved_model/qwen_techdrawing_merged_16bit"
    print(f"💾 Lưu merged 16-bit vào: {MERGED_DIR}")
    model.save_pretrained_merged(MERGED_DIR, tokenizer)
    print("✅ Xong!")

    # Hoặc push merged lên HF
    # model.push_to_hub_merged(f"{HF_USERNAME}/{HF_REPO_NAME}-merged", tokenizer, token=HF_TOKEN)
else:
    print("ℹ️  Bỏ qua lưu 16-bit (SAVE_16BIT=False). Đặt True để kích hoạt.")

## Bước 12: Load Model Đã Fine-tune (Khi Cần Dùng Lại)
*Dùng cell này để load model đã lưu từ Drive, không cần train lại.*

In [ ]:
LOAD_SAVED = False  # ← Đặt True để load model từ Drive

if LOAD_SAVED:
    from unsloth import FastVisionModel

    # Option 1: Load từ Google Drive
    model, tokenizer = FastVisionModel.from_pretrained(
        model_name = SAVE_DIR,
        load_in_4bit = True,
    )

    # Option 2: Load từ Hugging Face Hub
    # model, tokenizer = FastVisionModel.from_pretrained(
    #     model_name = f"{HF_USERNAME}/{HF_REPO_NAME}",
    #     load_in_4bit = True,
    #     token = HF_TOKEN,
    # )

    FastVisionModel.for_inference(model)
    print(f"✅ Đã load model từ: {SAVE_DIR}")
else:
    print("ℹ️  Đặt LOAD_SAVED=True để load model đã lưu.")

---
## 📋 Tóm Tắt Workflow

| Bước | Mô tả | Ghi chú |
|------|--------|----------|
| 1 | Cài đặt thư viện | Chạy 1 lần |
| 2 | Mount Drive & cấu hình path | Điền `HF_USERNAME`, `HF_TOKEN` |
| 3 | Tạo dữ liệu mẫu | Chạy 1 lần để tạo template |
| 4 | Xem trước dataset | Kiểm tra ảnh + mô tả |
| 5-6 | Load model + LoRA | ~5-10 phút |
| 7 | Chuẩn bị dataset | Auto |
| 8 | Inference TRƯỚC train | Baseline |
| 9 | **Train** | Phần chính |
| 10 | Inference SAU train | So sánh kết quả |
| 11 | **Lưu** lên Drive + HF | LoRA adapters |
| 12 | Load lại khi cần | `LOAD_SAVED=True` |

### 📌 Cấu trúc `annotations.json`
```json
[
  {
    "image": "ten_anh.jpg",
    "description": "Mô tả chi tiết bản vẽ kỹ thuật..."
  }
]
```

### 💡 Gợi ý mô tả chất lượng cao
Một mô tả tốt nên bao gồm:
- **Loại chi tiết**: trục, bánh răng, tấm, vỏ hộp...
- **Kích thước chính**: đường kính, chiều dài, chiều rộng, chiều cao
- **Dung sai** (nếu có): H7/k6, IT6...
- **Vật liệu**: C45, CT3, GX150...
- **Yêu cầu kỹ thuật**: độ nhám Ra, nhiệt luyện, mạ phủ...
- **Đặc điểm đặc biệt**: lỗ, rãnh, ren, then hoa...